## Week 4 Notebook 4: Agent eval
* evaluate agent tool trajectories 

In [2]:
# Load the same ground-truth data
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
# Load the FAQ docs and the minsearch index
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
# Create a lookup table
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

#### Running the Agent
* Reuse the ToyAIKit agent from week 1. It handles the agent loop and stores the full message history.

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [ ]:
# Define the search tool
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"},
    )

In [ ]:
# create the runner
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

The result contains:

* `last_message`: the final response
* `all_messages`: the full message history
* `cost`: the cost of all LLM calls in this run

In [ ]:
# Run it for one ground truth question:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [ ]:
# Look at the full message history
result.all_messages

In [12]:
# Extract the function name and arguments
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [13]:
# Run agent eval on one example:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late start course closed can I still start now enrollment open"}'}]

In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

# Save the A->Q->A' record and the trajectory:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,  # this field is what we evaluate with the LLM judge
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,  # this field lets the judge see how the agent got there
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result


{'question': 'I found this course late — can I still start it now, or is it already closed?',
 'answer_agent': 'Yes — you can still start the course now.\n\nAccording to the FAQ, you can join late and begin whenever you want. The videos and GitHub materials are available, and deadlines are listed on the course platform. If you want to work toward a certificate, just make sure to submit the required project while submissions are still being accepted.\n\nIf you want, I can also help you find the course materials or explain the weekly workflow.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late start course closed can I still start now enrollment open"}'}],
 'cost': Decimal('0.00125475'),
 'document': '74eb249bbf'}

Processing multiple questions

In [ ]:
# Create a function that processes one ground truth record:


def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Run it for a small sample (first 50 docs) in parallel:

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [17]:
df_agent = pd.DataFrame(agent_answers)

df_agent["cost"].sum()

Decimal('0.06225150')

Our judge can look at both:

* whether answer_agent matches answer_orig
* whether the tool calls look reasonable for the question

In [20]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I found this course late — can I still start i...,Yes — you can still start it now. The course m...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.0012075,74eb249bbf
1,Is it okay to join the course after it has alr...,Yes — you can still join the course after it h...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""jo...",0.0009555,74eb249bbf
2,"If I enroll now, will I still be able to get a...","Yes — you can still join, but to get a certifi...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""en...",0.00121875,74eb249bbf
3,What’s the deadline for submitting the project...,"To get the certificate in **LLM Zoomcamp**, yo...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""de...",0.001269,74eb249bbf
4,"Can I join the course anytime, or do I have to...","Yes, you can still join the course anytime. Bu...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""jo...",0.00102525,74eb249bbf


In [18]:
# save the results
df_agent.to_csv("data/agent-answers.csv", index=False)

## Judging answers and trajectories

A good trajectory is not just "many tool calls". A good trajectory uses
the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

- The search query is relevant to the user question
- The query includes the important keywords from the question
- The agent avoids duplicate searches with the same arguments
- If it searches more than once, the next query is a useful refinement
- It usually uses 1 search call
- 2-3 calls can be okay for harder questions
- More than 3 search calls needs a clear reason
- The tool calls support the final answer
- The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

The judge instructions

In [22]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Define the judge function:


In [ ]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry


def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [24]:
# test it on one agent result:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It clearly says the course can still be started now, and it preserves the key condition for receiving a certificate: the project must be submitted while submissions are still being accepted. Extra detail about course materials and homework is not inconsistent with the original answer.', answer_score='good', trajectory_reasoning="The tool query was relevant to the question and included useful keywords like 'late start', 'still start now', and 'closed enrollment FAQ'. Only one search was used, which is reasonable for this straightforward question, and it supported the final answer.", trajectory_score='good')

Running the agent judge
* Run the judge for all agent answers:

In [25]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [26]:
# use the same parallel processor:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# Split the results, create a dataframe & calculate cost for judges

In [29]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

df_agent_eval = pd.DataFrame(agent_evaluations)

calc_total_price(usages)

0.053829749999999996

In [30]:
# Check the answer scores
df_agent_eval["answer_score"].value_counts()

answer_score
good    49
bad      1
Name: count, dtype: int64

In [31]:
# check the tool calling trajectory scores
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [32]:
# save the agentic judge results
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)